# 🎬 IMDb Movie Data Analysis

A Python/Pandas exploratory analysis of 1,000 IMDb movies, focusing on movie ratings, revenue, genres, directors, runtime and audience votes.

**Tools:** Python · Pandas · NumPy · Matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_csv("imdb_data.csv")
df.head()

## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

## 2. Data Cleaning

In [ ]:
# Standardize numeric columns
numeric_cols = ["Rank", "Year", "Runtime", "Rating", "Votes", "Revenue", "Metascore"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Basic cleaning for text fields
for col in ["Title", "Genre", "Director"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df.info()

## 3. Rating & Revenue Analysis

In [ ]:
summary = df[["Rating", "Revenue", "Votes", "Runtime"]].describe()
summary

In [ ]:
# Top 10 highest-rated movies
top_rated = df[["Title", "Rating", "Votes"]].sort_values(
    ["Rating", "Votes"], ascending=[False, False]
).head(10)
top_rated

In [ ]:
# Top 10 movies by revenue
top_revenue = df[["Title", "Revenue", "Rating"]].sort_values(
    "Revenue", ascending=False
).head(10)
top_revenue

## 4. Genre Analysis

In [ ]:
# Each movie can contain multiple genres.
genre_counts = (
    df["Genre"]
    .dropna()
    .str.split(", ")
    .explode()
    .value_counts()
)

genre_counts.head(10)

In [ ]:
# Average rating by genre
genre_rating = (
    df.assign(Genre=df["Genre"].str.split(", "))
      .explode("Genre")
      .groupby("Genre")
      .agg(
          Movies=("Title", "count"),
          Avg_Rating=("Rating", "mean"),
          Avg_Revenue=("Revenue", "mean")
      )
      .sort_values("Avg_Rating", ascending=False)
)

genre_rating.head(10)

## 5. Director Analysis

In [ ]:
director_analysis = (
    df.groupby("Director")
      .agg(
          Movies=("Title", "count"),
          Avg_Rating=("Rating", "mean"),
          Total_Revenue=("Revenue", "sum")
      )
      .sort_values(["Movies", "Avg_Rating"], ascending=[False, False])
)

director_analysis.head(10)

## 6. Yearly Trends

In [ ]:
yearly = (
    df.groupby("Year")
      .agg(
          Movies=("Title", "count"),
          Avg_Rating=("Rating", "mean"),
          Avg_Revenue=("Revenue", "mean")
      )
      .sort_index()
)

yearly.tail(10)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(yearly.index, yearly["Movies"], marker="o")
plt.title("Number of Movies by Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

## 7. Rating vs Revenue

In [ ]:
corr = df[["Rating", "Revenue", "Votes", "Runtime"]].corr(numeric_only=True)
corr

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Rating"], df["Revenue"], alpha=0.5)
plt.title("Movie Rating vs Revenue")
plt.xlabel("IMDb Rating")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## 8. Key Takeaways

In [ ]:
# Portfolio-ready summary metrics
print("Movies analysed:", len(df))
print("Average IMDb rating:", round(df["Rating"].mean(), 2))
print("Median IMDb rating:", round(df["Rating"].median(), 2))
print("Highest rating:", df["Rating"].max())
print("Highest revenue:", round(df["Revenue"].max(), 2))
print("Most common genre:", genre_counts.index[0])
print("Most represented year:", int(yearly["Movies"].idxmax()))